# ERA5 Time-Series Modelling and Risk Classification
## Drought and Flood Risk 2000–2025

> **Note — initial exploration notebook:** The SPEI and API values used here are the raw approximations stored in `era5_2000_2025.parquet` (30-day rolling SPEI, API with k=0.85). Phase 3 (`phase3_index_eda.ipynb`) recomputes both indices properly (SPEI-6/12 via log-logistic distribution, API with k=0.92). The risk classifiers also use the initial **5-class scheme** (Low, Moderate, Elevated, High, Extreme); Phase 3 revised this to a **4-class scheme** aligned with McKee (1993) SPEI thresholds (Low, Moderate, High, Extreme). The flood composite formula and drought modifier rules were also revised in Phase 3 — see sections 12–13 for details. This notebook is preserved to document the exploratory baseline and to establish which univariate forecasting model works best for each index.

This notebook models four climate indices derived from the ERA5 dataset
(period 2000–2025, location 42.75°E / 9.25°N, Horn of Africa):

| Index | Description | Risk target |
|-------|-------------|-------------|
| **SPEI** | Standardized Precipitation-Evapotranspiration Index | Drought |
| **API** | Antecedent Precipitation Index | Drought |
| **SMI** | Soil Moisture Index | Flood |
| **Total Runoff** | Total runoff (m/day) | Flood |

**Approach**: six purely autoregressive time-series models are fitted on daily data,
compared, and used to generate forecasts at **1, 3, 7, and 14 days** ahead.
No external features (precipitation, temperature) are used as inputs — only lagged
values of each index itself. Risk scores are then derived from those forecasts using
the initial **5-class scheme**: *Low*, *Moderate*, *Elevated*, *High*, *Extreme*
(later revised to 4 classes in Phase 3).

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

sys.path.insert(0, "..")

from src.modeling import (
    TimeSeriesPreprocessor,
    check_stationarity,
    SARIMAForecaster,
    HoltWintersForecaster,
    RandomForestForecaster,
    XGBoostForecaster,
    LSTMForecaster,
    ProphetForecaster,
    DroughtRiskClassifier,
    FloodRiskClassifier,
    RiskLevel,
)

# ─── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 13})

RISK_COLORS = {
    RiskLevel.LOW:      "#2ecc71",
    RiskLevel.MODERATE: "#f1c40f",
    RiskLevel.ELEVATED: "#e67e22",
    RiskLevel.HIGH:     "#e74c3c",
    RiskLevel.EXTREME:  "#8e44ad",
}
RISK_LEGEND = [Patch(color=c, label=r.value) for r, c in RISK_COLORS.items()]

DATA_PATH = "../src/data/processed/era5_2000_2025.parquet"
TEST_START = "2023-01-01"
FORECAST_HORIZONS = [1, 3, 7, 14]  # days ahead
import os; os.makedirs("figures", exist_ok=True)

## 1 · Load and explore data

In [ ]:
# ── Daily data (all models operate at daily resolution) ──────────────────────
prep_d = TimeSeriesPreprocessor(DATA_PATH, resample_freq=None).load()
df_d = prep_d.df

print(f"Daily: {df_d.shape[0]} rows  ({df_d.index[0].date()} → {df_d.index[-1].date()})")
df_d[["spei", "api", "smi", "total_ro"]].describe().round(4)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
pairs = [
    ("spei",     "SPEI",         "royalblue",  "Drought index"),
    ("api",      "API (m)",      "darkorange",  "Antecedent precipitation"),
    ("smi",      "SMI (0–1)",    "seagreen",    "Soil moisture index"),
    ("total_ro", "Runoff (m/d)", "firebrick",   "Total runoff"),
]
for ax, (col, ylabel, color, title) in zip(axes, pairs):
    ax.plot(df_d.index, df_d[col], lw=0.7, color=color, alpha=0.85)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, loc="left", fontweight="bold")
    ax.axvline(pd.Timestamp(TEST_START), color="grey", ls="--", lw=1.2,
               label="train/test split")

axes[0].legend(loc="upper right", fontsize=9)
axes[-1].set_xlabel("Date")
fig.suptitle("ERA5 Climate Indices – daily values (2000–2025)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("figures/overview_timeseries.png", bbox_inches="tight")
plt.show()

## 2 · Stationarity analysis (ADF test)

A time series is *stationary* if its mean and variance do not change over time.
The **Augmented Dickey-Fuller (ADF)** test tests the null hypothesis H₀: *the series
contains a unit root (non-stationary)*. A p-value < 0.05 rejects H₀ → stationary.

ARIMA can handle non-stationarity via the differencing parameter `d`.
Holt-Winters requires no stationarity assumption thanks to exponential smoothing.

In [ ]:
TARGET_COLS = ["spei", "api", "smi", "total_ro"]

rows = []
for col in TARGET_COLS:
    res = check_stationarity(df_d[col])
    rows.append({
        "Index": col.upper(),
        "ADF statistic": round(res.statistic, 3),
        "p-value": round(res.p_value, 4),
        "Stationary (α=0.05)": "✔ Yes" if res.is_stationary else "✘ No",
        "Critical value 5%": round(res.critical_values["5%"], 3),
    })

pd.DataFrame(rows).set_index("Index")

## 3 · Seasonality analysis – ACF and PACF

The **Autocorrelation Function (ACF)** shows the correlation of the daily series with
itself at various lags (in days). The **Partial ACF (PACF)** corrects for intermediate
lags. Peaks at lag 7 (weekly) or lag 365 (annual) indicate seasonal patterns.
For short-horizon daily models, the first 30 lags are most informative for ARIMA
parameter selection.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12))

for i, col in enumerate(TARGET_COLS):
    series = df_d[col].dropna()
    plot_acf(series, lags=30, ax=axes[i, 0],
             title=f"ACF – {col.upper()} (daily)", zero=False)
    plot_pacf(series, lags=20, ax=axes[i, 1],
              title=f"PACF – {col.upper()} (daily)", zero=False, method="ywm")

plt.tight_layout()
plt.savefig("figures/acf_pacf.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10))
for ax, col in zip(axes, TARGET_COLS):
    series = df_d[col].dropna()
    result = seasonal_decompose(series, model="additive", period=365)
    ax.plot(result.seasonal, lw=0.8, color="steelblue", label="Annual seasonal component")
    ax.set_title(f"Seasonal component – {col.upper()}", fontsize=11, loc="left")
    ax.set_xlabel("")
    ax.legend(fontsize=9)

plt.suptitle("Additive seasonal decomposition (daily data, period=365 days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/seasonal_decomposition.png", bbox_inches="tight")
plt.show()

## 4 · Train/test split

In [ ]:
train_d, test_d = prep_d.train_test_split(TEST_START)

print(f"Daily – train: {len(train_d)} | test: {len(test_d)}")
print(f"Training period: {train_d.index[0].date()} → {train_d.index[-1].date()}")
print(f"Test period:     {test_d.index[0].date()}  → {test_d.index[-1].date()}")

## 5 · Model 1 – ARIMA (daily, no seasonal component)

**AutoRegressive Integrated Moving Average** (ARIMA) combines auto-regression (p),
differencing (d), and moving average (q).
On daily data, SARIMA with annual seasonality (s=365) is computationally impractical
due to the large number of parameters to estimate. ARIMA without a seasonal component
is used here as a fast baseline; the annual cycle is captured by the Holt-Winters
and lag-based models in later sections.

Chosen order per index:

| Index | (p,d,q) | Reason |
|-------|---------|--------|
| SPEI     | (1,1,1) | High lag-1 autocorrelation, slow mean-reversion |
| API      | (1,1,1) | Similar autocorrelation profile |
| SMI      | (1,1,1) | Strong soil moisture persistence |
| Total Runoff | (1,1,0) | Highly skewed distribution, simple AR structure |

> **Expected result**: all R² values are negative — ARIMA without a seasonal component
> cannot capture the strong annual dry/wet cycle and therefore fails to beat the mean.
> This is expected and motivates the seasonal approaches in sections 6–10.

In [ ]:
ARIMA_PARAMS = {
    "spei":     {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    "api":      {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    "smi":      {"order": (1, 1, 1), "seasonal_order": (0, 0, 0, 0)},
    "total_ro": {"order": (1, 1, 0), "seasonal_order": (0, 0, 0, 0)},
}

sarima_models = {}
sarima_metrics = {}

for col, params in ARIMA_PARAMS.items():
    print(f"  Fitting ARIMA for {col.upper()} ...", end=" ")
    model = SARIMAForecaster(target_column=col, **params)
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    sarima_models[col] = model
    sarima_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
for ax, col in zip(axes, TARGET_COLS):
    model = sarima_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax.plot(train_d.index[-90:], train_d[col].iloc[-90:],
            color="steelblue", lw=1.2, label="Training (last 90 days)")
    ax.plot(test_d.index[:90], test_d[col].iloc[:90],
            color="black", lw=1.5, ls="--", label="Actual (test, 90 days)")
    ax.plot(test_d.index[:90], pred.iloc[:90],
            color="tomato", lw=1.5, label="ARIMA forecast")
    ax.axvline(pd.Timestamp(TEST_START), color="grey", ls=":", lw=1)
    ax.set_title(f"ARIMA – {col.upper()}", loc="left")
    ax.legend(fontsize=9)

plt.suptitle("ARIMA: forecasts vs. actual values (first 90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/arima_results.png", bbox_inches="tight")
plt.show()

## 6 · Model 2 – Holt-Winters (daily, additive annual seasonality)

Holt-Winters applies exponential smoothing to level, trend, and a seasonal component.
With `seasonal_periods=365` the annual dry/wet cycle is explicitly modelled —
a strong assumption for the Horn of Africa. The additive model is appropriate because
seasonal amplitude is roughly constant over the training period.

The model requires no stationarity assumption and adapts smoothly to recent values.

> **Expected result**: Holt-Winters works reasonably for API and SMI but fails
> catastrophically for SPEI (observed R²=−1420). SPEI can sustain prolonged negative
> excursions during multi-year droughts that violate the additive seasonal assumption;
> the model extrapolates the training-period amplitude into the test period, producing
> extreme overshooting. This is an important signal that SPEI requires a model capable
> of handling non-stationary dynamics.

In [ ]:
hw_models = {}
hw_metrics = {}

for col in TARGET_COLS:
    print(f"  Fitting HoltWinters for {col.upper()} ...", end=" ")
    model = HoltWintersForecaster(
        target_column=col,
        seasonal_periods=365,
        trend="add",
        seasonal="add",
    )
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    hw_models[col] = model
    hw_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
for ax, col in zip(axes, TARGET_COLS):
    model = hw_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax.plot(train_d.index[-90:], train_d[col].iloc[-90:],
            color="steelblue", lw=1.2, label="Training (last 90 days)")
    ax.plot(test_d.index[:90], test_d[col].iloc[:90],
            color="black", lw=1.5, ls="--", label="Actual")
    ax.plot(test_d.index[:90], pred.iloc[:90],
            color="mediumseagreen", lw=1.5, label="Holt-Winters forecast")
    ax.set_title(f"Holt-Winters – {col.upper()}", loc="left")
    ax.legend(fontsize=9)

plt.suptitle("Holt-Winters: forecasts vs. actual values (first 90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/holtwinters_results.png", bbox_inches="tight")
plt.show()

## 7 · Model 3 – Random Forest (daily, lag features)

Random Forest recasts the forecasting problem as supervised regression by supplying
the time series with *lag features* — lagged values of the index itself. No external
features (precipitation, temperature) are used.

Lags used: **[1, 3, 7, 14, 365]** days.
- Lags 1–14 capture short-term autocorrelation for 1–14 day ahead forecasting.
- Lag 365 encodes the annual dry/wet seasonal cycle (Jijiga bimodal rainy seasons).

Cyclic date encodings (sin/cos) are **not** added; seasonal information is entirely
encoded by lag_365.

In [ ]:
rf_models = {}
rf_metrics = {}

for col in TARGET_COLS:
    print(f"  Fitting RandomForest for {col.upper()} ...", end=" ")
    model = RandomForestForecaster(
        target_column=col,
        lags=[1, 3, 7, 14, 365],
        n_estimators=200,
        random_state=42,
        include_cyclical=False,
    )
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    rf_models[col] = model
    rf_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
for ax, col in zip(axes, TARGET_COLS):
    model = rf_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax.plot(train_d.index[-90:], train_d[col].iloc[-90:],
            color="steelblue", lw=1.2, label="Training (last 90 days)")
    ax.plot(test_d.index[:90], test_d[col].iloc[:90],
            color="black", lw=1.5, ls="--", label="Actual")
    ax.plot(test_d.index[:90], pred.iloc[:90],
            color="darkorchid", lw=1.5, label="Random Forest forecast")
    ax.set_title(f"Random Forest – {col.upper()}", loc="left")
    ax.legend(fontsize=9)

plt.suptitle("Random Forest: forecasts vs. actual values (first 90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/rf_results.png", bbox_inches="tight")
plt.show()

In [ ]:
# Feature importance for SPEI (representative example)
fi = rf_models["spei"].feature_importances
fig, ax = plt.subplots(figsize=(8, 4))
fi.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Feature importance – Random Forest SPEI", fontweight="bold")
ax.set_xlabel("Importance (mean decrease impurity)")
plt.tight_layout()
plt.savefig("figures/rf_feature_importance.png", bbox_inches="tight")
plt.show()

## 8 · Model 4 – XGBoost (daily, gradient boosting)

**XGBoost** uses the same lag features as Random Forest (`[1, 3, 7, 14, 365]` days)
but builds trees sequentially: each tree corrects the residual error of the previous one.
This typically yields lower bias on tabular time-series data at the cost of slightly
more training time.

No cyclic date encodings are added — seasonal information is fully captured by lag_365.

In [ ]:
xgb_models = {}
xgb_metrics = {}

for col in TARGET_COLS:
    print(f"  Fitting XGBoost for {col.upper()} ...", end=" ")
    model = XGBoostForecaster(
        target_column=col,
        lags=[1, 3, 7, 14, 365],
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        include_cyclical=False,
    )
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    xgb_models[col] = model
    xgb_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
for i, col in enumerate(TARGET_COLS):
    # Left: forecast vs actual
    ax_left = axes[i, 0]
    model = xgb_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax_left.plot(train_d.index[-90:], train_d[col].iloc[-90:],
                 color="steelblue", lw=1.2, label="Training (90 days)")
    ax_left.plot(test_d.index[:90], test_d[col].iloc[:90],
                 color="black", lw=1.5, ls="--", label="Actual")
    ax_left.plot(test_d.index[:90], pred.iloc[:90],
                 color="goldenrod", lw=1.5, label="XGBoost")
    ax_left.set_title(f"XGBoost – {col.upper()}", loc="left")
    ax_left.legend(fontsize=8)

    # Right: feature importance
    ax_right = axes[i, 1]
    fi = xgb_models[col].feature_importances
    fi.plot.barh(ax=ax_right, color="goldenrod", edgecolor="white")
    ax_right.set_title(f"Feature importance – {col.upper()}", loc="left")
    ax_right.set_xlabel("Importance")

plt.suptitle("XGBoost: forecasts and feature importances (90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/xgb_results.png", bbox_inches="tight")
plt.show()

## 9 · Model 5 – LSTM (daily, sequential deep learning)

**Long Short-Term Memory (LSTM)** is a recurrent neural network that learns long-range
dependencies in time series via gating mechanisms (input, forget, and output gates).
The model sees a window of the last `lookback=30` days and predicts the next daily value.

Architecture:
```
Input (30 × 1) → LSTM(64) → Dropout(0.1) → Dense(1)
```

Values are Min-Max normalised before training and inverse-transformed back to the
original scale after prediction.

In [ ]:
lstm_models = {}
lstm_metrics = {}

for col in TARGET_COLS:
    print(f"  Fitting LSTM for {col.upper()} ...", end=" ")
    model = LSTMForecaster(
        target_column=col,
        lookback=30,
        units=64,
        epochs=20,
        batch_size=64,
        random_state=42,
    )
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    lstm_models[col] = model
    lstm_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
for ax, col in zip(axes, TARGET_COLS):
    model = lstm_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax.plot(train_d.index[-90:], train_d[col].iloc[-90:],
            color="steelblue", lw=1.2, label="Training (last 90 days)")
    ax.plot(test_d.index[:90], test_d[col].iloc[:90],
            color="black", lw=1.5, ls="--", label="Actual")
    ax.plot(pred.index[:90], pred.iloc[:90],
            color="darkcyan", lw=1.5, label="LSTM forecast")
    ax.set_title(f"LSTM – {col.upper()}", loc="left")
    ax.legend(fontsize=9)

plt.suptitle("LSTM: forecasts vs. actual values (first 90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/lstm_results.png", bbox_inches="tight")
plt.show()

## 10 · Model 6 – Prophet (daily, trend + annual seasonality)

**Facebook Prophet** decomposes the time series into a flexible trend (piecewise linear
with automatic changepoints) and a Fourier seasonal component.
With `yearly_seasonality=True` the model learns the annual dry/wet cycle directly
without manual period specification.

Advantages for climate data:
- Explicit modelling of the annual cycle without hand-tuning a period parameter.
- Robust to missing values and outliers.
- No stationarity requirement.

In [ ]:
prophet_models = {}
prophet_metrics = {}

for col in TARGET_COLS:
    print(f"  Fitting Prophet for {col.upper()} ...", end=" ")
    model = ProphetForecaster(
        target_column=col,
        yearly_seasonality=True,
        weekly_seasonality=False,
        changepoint_prior_scale=0.05,
    )
    model.fit(train_d[col])
    pred = model.predict_in_sample(test_d[col])
    metrics = model.evaluate(test_d[col], pred)
    prophet_models[col] = model
    prophet_metrics[col] = metrics
    print(f"RMSE={metrics.rmse:.4f}  MAE={metrics.mae:.4f}  R²={metrics.r2:.3f}")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=False)
for ax, col in zip(axes, TARGET_COLS):
    model = prophet_models[col]
    pred  = model.predict_in_sample(test_d[col])
    ax.plot(train_d.index[-90:], train_d[col].iloc[-90:],
            color="steelblue", lw=1.2, label="Training (last 90 days)")
    ax.plot(test_d.index[:90], test_d[col].iloc[:90],
            color="black", lw=1.5, ls="--", label="Actual")
    ax.plot(test_d.index[:90], pred.iloc[:90],
            color="coral", lw=1.5, label="Prophet forecast")
    ax.set_title(f"Prophet – {col.upper()}", loc="left")
    ax.legend(fontsize=9)

plt.suptitle("Prophet: forecasts vs. actual values (first 90 test days)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/prophet_results.png", bbox_inches="tight")
plt.show()

## 11 · Baseline comparison – all 6 models

All six baseline models are compared on the same test set (2023–2025).
This overview determines which model per climate index to carry forward as
the reference benchmark to the feature-engineering and XGBoost phase (Phase 4–5).

In [ ]:
rows = []
for col in TARGET_COLS:
    for model_name, metrics_dict in [
        ("ARIMA",        sarima_metrics),
        ("Holt-Winters", hw_metrics),
        ("RandomForest", rf_metrics),
        ("XGBoost",      xgb_metrics),
        ("LSTM",         lstm_metrics),
        ("Prophet",      prophet_metrics),
    ]:
        m = metrics_dict[col]
        rows.append({
            "Index": col.upper(),
            "Model": model_name,
            "RMSE": round(m.rmse, 5),
            "MAE":  round(m.mae, 5),
            "R²":   round(m.r2, 3),
        })

comparison_df = pd.DataFrame(rows).set_index(["Index", "Model"])
comparison_df.style.highlight_min(subset=["RMSE", "MAE"], color="lightgreen") \
                   .highlight_max(subset=["R²"],           color="lightgreen")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric in zip(axes, ["RMSE", "MAE", "R²"]):
    pivot = comparison_df[metric].unstack("Model")
    pivot.plot.bar(ax=ax, edgecolor="white")
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Index")
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=0)

plt.suptitle("Model comparison on test set (2023–2025)", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("figures/model_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Select best model per index based on RMSE
best_models = {}
model_lookup = {
    "ARIMA":        sarima_models,
    "Holt-Winters": hw_models,
    "RandomForest": rf_models,
    "XGBoost":      xgb_models,
    "LSTM":         lstm_models,
    "Prophet":      prophet_models,
}

for col in TARGET_COLS:
    col_df = comparison_df.loc[col.upper(), "RMSE"]
    best_name = col_df.idxmin()
    print(f"{col.upper():12s} → best baseline: {best_name}  "
          f"(RMSE={col_df[best_name]:.5f})")
    best_models[col] = model_lookup[best_name][col]

## 12 · Drought risk classification

> **Design note — revised in Phase 3:** This classifier uses the initial **5-class scheme**
> (Low, Moderate, Elevated, High, Extreme) and applies API, SMI, and Total Runoff as
> secondary modifiers. Phase 3 (`phase3_index_eda.ipynb`) revised this to a **4-class
> scheme** (Low, Moderate, High, Extreme) aligned with McKee (1993) thresholds, and
> **removed API from the drought modifier** because near-zero API values during dry seasons
> fire the modifier spuriously on days that are already dry but not genuinely at elevated
> drought risk. The 5-class version is preserved here as the initial design.

The `DroughtRiskClassifier` combines SPEI, API, SMI, and Total Runoff:

1. **SPEI** (primary) – classified according to McKee et al. (1993) extended to 5 classes:
   - ≥ −0.5 → *Low* | −1.0 to −0.5 → *Moderate*
   - −1.5 to −1.0 → *Elevated* | −2.0 to −1.5 → *High* | < −2.0 → *Extreme*
2. **API, SMI, Total Runoff** (modifiers) – if any of these indicators falls below the
   training 25th percentile, risk is elevated by one class. The modifier is skipped when
   SPEI is already Low or Extreme.

In [ ]:
drought_clf = DroughtRiskClassifier(api_low_percentile=0.25)
drought_clf.fit(train_d)

# Classify the full daily series
drought_risk_daily = drought_clf.classify(df_d)

# Summary by risk class
counts = drought_risk_daily.value_counts()
pct    = (counts / len(drought_risk_daily) * 100).round(1)
summary = pd.DataFrame({"Days": counts, "%": pct})
summary.index = [r.value for r in summary.index]
print("Drought risk – distribution over full period (2000–2025):")
summary

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Time series with threshold lines
ax = axes[0]
ax.plot(df_d.index, df_d["spei"], lw=0.8, color="royalblue", label="SPEI")
ax.axhline(-0.5, color="orange",  ls="--", lw=0.8, alpha=0.7)
ax.axhline(-1.0, color="darkorange", ls="--", lw=0.8, alpha=0.7)
ax.axhline(-1.5, color="red", ls="--", lw=0.8, alpha=0.7)
ax.axhline(-2.0, color="purple", ls="--", lw=0.8, alpha=0.7)
ax.set_ylabel("SPEI")
ax.set_title("SPEI with drought risk thresholds", loc="left")
ax.legend(["SPEI", "Moderate (−0.5)", "Elevated (−1.0)",
           "High (−1.5)", "Extreme (−2.0)"], fontsize=9)

# Risk as numeric time series coloured by level
ax2 = axes[1]
risk_numeric = drought_risk_daily.apply(lambda r: r.numeric)
for level in RiskLevel:
    mask = drought_risk_daily == level
    ax2.scatter(
        df_d.index[mask], risk_numeric[mask],
        s=1.5, color=RISK_COLORS[level], label=level.value
    )
ax2.set_yticks(range(5))
ax2.set_yticklabels([r.value for r in RiskLevel], fontsize=8)
ax2.set_title("Daily drought risk (2000–2025)", loc="left")
ax2.legend(handles=RISK_LEGEND, fontsize=8, ncol=5, loc="upper right")

plt.tight_layout()
plt.savefig("figures/drought_risk_historical.png", bbox_inches="tight")
plt.show()

## 13 · Flood risk classification

> **Design note — revised in Phase 3:** This classifier uses an **equal-weight composite
> that includes SPEI** as one of the four inputs (0.25 × norm_SPEI + 0.25 × norm_API +
> 0.25 × norm_SMI + 0.25 × norm_runoff). Phase 3 (`phase3_index_eda.ipynb`) **removed
> SPEI** from the flood composite entirely, because negative SPEI during dry spells that
> precede sudden flash floods suppresses the composite score at precisely the wrong
> moment. The revised weights are 0.40 × API + 0.35 × SMI + 0.25 × runoff.
> The equal-weight version is preserved here as the initial design.

The `FloodRiskClassifier` combines SPEI, API, SMI, and Total Runoff via a weighted
composite score:

    flood_score = 0.25 × norm_SPEI + 0.25 × norm_API + 0.25 × norm_SMI + 0.25 × norm_total_ro

All indicators are Min-Max normalised on training data.
Risk thresholds are derived from empirical percentiles of the composite score.

In [ ]:
flood_clf = FloodRiskClassifier()
flood_clf.fit(train_d)

flood_risk_daily = flood_clf.classify(df_d)

counts_f = flood_risk_daily.value_counts()
pct_f    = (counts_f / len(flood_risk_daily) * 100).round(1)
summary_f = pd.DataFrame({"Days": counts_f, "%": pct_f})
summary_f.index = [r.value for r in summary_f.index]
print("Flood risk – distribution over full period (2000–2025):")
summary_f

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# SMI and runoff time series
ax = axes[0]
ax.plot(df_d.index, df_d["smi"], lw=0.8, color="seagreen", label="SMI", alpha=0.8)
ax2b = ax.twinx()
ax2b.fill_between(df_d.index, df_d["total_ro"] * 1000, alpha=0.4,
                  color="steelblue", label="Total Runoff (×1000)")
ax2b.set_ylabel("Runoff ×10³ m/d", color="steelblue")
ax.set_ylabel("SMI (0–1)")
ax.set_title("SMI and Total Runoff", loc="left")
ax.legend(loc="upper left", fontsize=9)
ax2b.legend(loc="upper right", fontsize=9)

# Risk as numeric time series coloured by level
ax3 = axes[1]
risk_numeric_f = flood_risk_daily.apply(lambda r: r.numeric)
for level in RiskLevel:
    mask = flood_risk_daily == level
    ax3.scatter(
        df_d.index[mask], risk_numeric_f[mask],
        s=1.5, color=RISK_COLORS[level], label=level.value
    )
ax3.set_yticks(range(5))
ax3.set_yticklabels([r.value for r in RiskLevel], fontsize=8)
ax3.set_title("Daily flood risk (2000–2025)", loc="left")
ax3.legend(handles=RISK_LEGEND, fontsize=8, ncol=5, loc="upper right")

plt.tight_layout()
plt.savefig("figures/flood_risk_historical.png", bbox_inches="tight")
plt.show()

In [ ]:
# Average risk per month (numeric)
df_risk = pd.DataFrame({
    "drought":  drought_risk_daily.apply(lambda r: r.numeric),
    "flood":    flood_risk_daily.apply(lambda r: r.numeric),
    "month":    df_d.index.month,
})
monthly_risk = df_risk.groupby("month")[["drought", "flood"]].mean()
monthly_risk.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                       "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(10, 5))
monthly_risk.plot.bar(ax=ax, color=["royalblue", "tomato"],
                      edgecolor="white", width=0.7)
ax.set_yticks(range(5))
ax.set_yticklabels([r.value for r in RiskLevel], fontsize=9)
ax.set_xlabel("Month")
ax.set_title("Average risk per calendar month (2000–2025)", fontweight="bold")
ax.legend(["Drought risk", "Flood risk"])
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig("figures/seasonal_risk.png", bbox_inches="tight")
plt.show()

## 14 · Future forecast – 1, 3, 7, and 14 days ahead

The best baseline model per index generates a **14-step daily forecast** from the
end of the training period (2022-12-31). The horizon values at the four specific
time steps are then extracted:

| Horizon | Description |
|---------|-------------|
| **+1 day**  | 2023-01-01 – high confidence |
| **+3 days** | 2023-01-03 – 3-day outlook |
| **+7 days** | 2023-01-07 – weekly forecast |
| **+14 days**| 2023-01-14 – 2-week forecast, greater uncertainty |

> **Note**: Because the forecast starts at 2022-12-31 (end of training), the forecast
> window overlaps with the beginning of the test set. The actual values are known,
> making this a verifiable out-of-sample evaluation.

> **Next step**: Phase 4 (`phase4_feature_engineering.ipynb`) constructs a multi-variable
> feature matrix with lag features across all ERA5 indices and meteorological variables.
> Phase 5 (`phase5_xgboost.ipynb`) trains XGBoost classifiers that predict risk level
> directly at each forecast horizon, replacing this univariate pipeline.

In [ ]:
# Generate 14-day forecast with the best model per index
future_14 = {}
for col in TARGET_COLS:
    future_14[col] = best_models[col].predict(steps=14)

future_df = pd.DataFrame(future_14)
future_df.index.name = "date"

# Extract the four specific horizons (0-based indexing: day 1 = index 0)
HORIZON_IDX = {h: h - 1 for h in FORECAST_HORIZONS}  # {1:0, 3:2, 7:6, 14:13}

horizon_rows = []
for h, idx in HORIZON_IDX.items():
    row = {"Horizon": f"+{h} day{'s' if h > 1 else ''}",
           "Date": future_df.index[idx].strftime("%Y-%m-%d")}
    for col in TARGET_COLS:
        row[col.upper()] = round(future_df[col].iloc[idx], 5)
    horizon_rows.append(row)

horizon_table = pd.DataFrame(horizon_rows).set_index("Horizon")
print(f"Forecast period: {future_df.index[0].date()} → {future_df.index[-1].date()}")
horizon_table

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 13), sharex=False)
COLORS = ["royalblue", "darkorange", "seagreen", "firebrick"]
MARKER_COLORS = {"1": "#e74c3c", "3": "#e67e22", "7": "#f1c40f", "14": "#8e44ad"}

for ax, col, color in zip(axes, TARGET_COLS, COLORS):
    hist = df_d[col].iloc[-60:]
    pred = future_df[col]
    ax.plot(hist.index, hist, color=color, lw=1.4, label="Historical (60 days)")
    ax.plot(pred.index, pred, color=color, lw=1.8, ls="--", alpha=0.7, label="Forecast (14 days)")
    ax.axvline(df_d.index[-1], color="grey", ls=":", lw=1.2, label="Forecast start")

    # Mark the four horizons
    for h, idx in HORIZON_IDX.items():
        ax.scatter(pred.index[idx], pred.iloc[idx],
                   s=80, zorder=5, color=MARKER_COLORS[str(h)],
                   label=f"+{h}d")

    ax.set_title(col.upper(), loc="left", fontweight="bold")
    ax.legend(fontsize=8, ncol=4)

plt.suptitle("14-day forecast – best model per index (horizons marked)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("figures/future_forecast.png", bbox_inches="tight")
plt.show()

In [ ]:
# Risk classification at each of the four horizons
risk_rows = []
for h, idx in HORIZON_IDX.items():
    spei_val = future_df["spei"].iloc[idx]
    api_val  = future_df["api"].iloc[idx]
    smi_val  = future_df["smi"].iloc[idx]
    ro_val   = future_df["total_ro"].iloc[idx]

    drought_risk = drought_clf.classify_from_predictions(
        spei_pred=pd.Series([spei_val]),
        api_pred=pd.Series([api_val]),
        smi_pred=pd.Series([smi_val]),
        total_ro_pred=pd.Series([ro_val]),
    ).iloc[0]

    flood_risk = flood_clf.classify_from_predictions(
        spei_pred=pd.Series([spei_val]),
        api_pred=pd.Series([api_val]),
        smi_pred=pd.Series([smi_val]),
        total_ro_pred=pd.Series([ro_val]),
    ).iloc[0]

    risk_rows.append({
        "Horizon":       f"+{h} day{'s' if h > 1 else ''}",
        "Date":          future_df.index[idx].strftime("%Y-%m-%d"),
        "SPEI (pred)":   round(spei_val, 3),
        "API (pred)":    round(api_val, 6),
        "Drought risk":  drought_risk.value,
        "SMI (pred)":    round(smi_val, 3),
        "Runoff (pred)": round(ro_val, 8),
        "Flood risk":    flood_risk.value,
    })

risk_forecast_df = pd.DataFrame(risk_rows).set_index("Horizon")
print("Risk classification per forecast horizon:")
risk_forecast_df

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
horizons_labels = [f"+{h}d" for h in FORECAST_HORIZONS]

# Extract risk levels as numeric values
drought_nums = []
flood_nums   = []
drought_colors = []
flood_colors   = []
for row in risk_rows:
    dr = next(r for r in RiskLevel if r.value == row["Drought risk"])
    fr = next(r for r in RiskLevel if r.value == row["Flood risk"])
    drought_nums.append(dr.numeric + 1)
    flood_nums.append(fr.numeric + 1)
    drought_colors.append(RISK_COLORS[dr])
    flood_colors.append(RISK_COLORS[fr])

for ax, nums, colors, title in [
    (axes[0], drought_nums, drought_colors, "Drought risk per horizon"),
    (axes[1], flood_nums,   flood_colors,   "Flood risk per horizon"),
]:
    ax.bar(horizons_labels, nums, color=colors, edgecolor="white", width=0.5)
    ax.set_yticks(range(1, 6))
    ax.set_yticklabels([r.value for r in RiskLevel], fontsize=9)
    ax.set_xlabel("Forecast horizon")
    ax.set_title(title, fontweight="bold", loc="left")
    ax.legend(handles=RISK_LEGEND, fontsize=8, ncol=5, loc="upper right")

plt.tight_layout()
plt.savefig("figures/future_risk.png", bbox_inches="tight")
plt.show()

## 15 · Conclusion – Baseline phase

### Results: model comparison on test set (2023–2025)

| Model | SPEI R² | API R² | SMI R² | Runoff R² |
|-------|---------|--------|--------|-----------|
| **ARIMA** | −0.131 | −0.403 | −0.695 | −0.017 |
| **Holt-Winters** | **−1420** | 0.167 | 0.045 | −0.503 |
| **Random Forest** | 0.993 | 0.869 | 0.934 | −0.352 |
| **XGBoost** | **0.993** | **0.878** | 0.933 | ~0.000 |
| **LSTM** | 0.984 | 0.879 | **0.943** | 0.002 |
| **Prophet** | 0.292 | 0.198 | 0.343 | −0.048 |

**Best model per index (by RMSE)**: SPEI → XGBoost, API → XGBoost, SMI → LSTM, Total Runoff → ARIMA*.

*Total Runoff caveat: ARIMA achieves the lowest absolute RMSE only because all models
predict near-zero values at this semi-arid grid point. With R²=−0.017, ARIMA is
still slightly worse than predicting the mean. Total runoff is structurally
unforecastable from local ERA5 precipitation alone — upstream Wabi Shabelle river
dynamics are the dominant driver of flood-relevant runoff events.

### Key findings

1. **Lag-based ML models (RF, XGBoost, LSTM) vastly outperform statistical models for
   SPEI, API, and SMI.** The lag_365 feature — encoding "what was the index value exactly
   one year ago" — is the primary driver. This implicitly captures the bimodal rainy
   season of Jijiga (April–May, October–November) without any calendar encoding.

2. **Holt-Winters catastrophically fails for SPEI** (R²=−1420): the additive seasonal
   model extrapolates the training-period amplitude into the test period, but SPEI's
   non-stationarity — sustained multi-year droughts like 2022–23 — violates the
   additive assumption and causes extreme overshooting.

3. **ARIMA is uniformly poor** (all R² < 0): without a seasonal component on daily data,
   no differencing order can reconstruct the strong annual cycle.

4. **XGBoost slightly edges out Random Forest** (SPEI: 0.993 vs 0.993 — tie on RMSE;
   API: R²=0.878 vs 0.869). The sequential boosting advantage is marginal here,
   consistent with the high autocorrelation structure of these indices making RF
   nearly as effective.

5. **LSTM wins on SMI** (R²=0.943 vs RF 0.934, XGBoost 0.933): the recurrent
   structure better captures the slower soil moisture dynamics. However, the gains
   are small and LSTM takes significantly longer to train.

6. **Prophet is a moderate compromise**: R²≈0.29–0.34 for SPEI and SMI, compared to
   ≈0.99 for ML models. Suitable only when trend + seasonality decomposition
   interpretability is the primary requirement.

### Comparison of the six baseline models

| Model | Strength | Limitation |
|-------|---------|-----------|
| **ARIMA** | Fast, parametric, interpretable | No seasonal component on daily data; fails to beat mean |
| **Holt-Winters** | Explicit annual cycle; adaptive level | Additive assumption breaks for non-stationary series; catastrophic on SPEI |
| **Random Forest** | Non-linear, robust, fast | Recursive prediction: error compounds across steps 2–14 |
| **XGBoost** | Lower bias via sequential boosting | Same recursive error compounding |
| **LSTM** | Best SMI performance; learns long-range memory | Slower training; requires scaling; sensitive to hyperparameters |
| **Prophet** | Transparent trend + seasonal decomposition | No autoregression; misses day-to-day variability |

### Limitations of this univariate baseline
- Each index is modelled **independently and autoregressively** — no cross-index or
  meteorological covariate information is used. Phase 4–5 address this with a
  multi-variable lag feature matrix.
- The `predict_in_sample` evaluation uses real observed lag values from the test set,
  giving RF/XGBoost/LSTM an **information advantage** over a true real-time deployment.
  Genuine multi-step-ahead RMSE (recursive prediction from a single launch date) would
  be higher.
- The SPEI column used here is a 30-day rolling approximation from the raw parquet.
  Phase 3 recomputes SPEI-6 properly via the log-logistic distribution, which changes
  the exact index values and therefore the risk class distributions.